In [ ]:
"""
Pupil Size Data Cleaning Pipeline
==================================
1. Synthetic data generation (clean + noisy pairs)
2. CNN model for signal denoising
3. Real-world data training/test split structure

Dependencies: numpy, torch, matplotlib, scikit-learn
Install: pip install torch numpy matplotlib scikit-learn
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


# ─────────────────────────────────────────────
# 1. SYNTHETIC DATA GENERATION
# ─────────────────────────────────────────────

def generate_pupil_signal(n: int = 300, n_flashes: int = None, rng: np.random.Generator = None) -> np.ndarray:
    """
    Generate a single clean pupil size signal.

    The pupil rests near a baseline, drops sharply on a light flash
    (pupillary light reflex), then recovers with an exponential curve.

    Args:
        n:         Length of the time series.
        n_flashes: Number of light flashes. Defaults to 1–3 random flashes.
        rng:       NumPy random generator (for reproducibility).

    Returns:
        signal: 1-D float32 array of shape (n,), values in [0, 1].
    """
    if rng is None:
        rng = np.random.default_rng()

    signal = np.ones(n, dtype=np.float32)  # baseline = 1.0

    if n_flashes is None:
        n_flashes = rng.integers(1, 4)

    # Space flashes so they don't overlap too much
    flash_positions = rng.integers(int(n * 0.05), int(n * 0.85), size=n_flashes)
    flash_positions = np.sort(flash_positions)

    for pos in flash_positions:
        drop_depth   = rng.uniform(0.3, 0.7)          # how much the pupil shrinks
        recovery_tau = rng.uniform(n * 0.05, n * 0.2) # recovery time constant

        t = np.arange(n - pos)
        recovery = 1.0 - drop_depth * np.exp(-t / recovery_tau)

        # Apply recovery from flash position onward
        signal[pos:] = np.maximum(signal[pos:], recovery.astype(np.float32))
        signal[pos]  = 1.0 - drop_depth  # sharp drop at flash

    # Small baseline drift
    drift = rng.uniform(-0.05, 0.05) * np.linspace(0, 1, n)
    signal = np.clip(signal + drift.astype(np.float32), 0.0, 1.2)

    return signal


def add_artifacts(signal: np.ndarray, rng: np.random.Generator = None) -> np.ndarray:
    """
    Add realistic artifacts to a clean pupil signal.

    Artifact types:
        - Gaussian noise       (sensor noise)
        - Blink artifacts      (sharp transient dips to ~0)
        - Spike artifacts      (brief large spikes up or down)
        - Slow baseline drift  (head movement / lighting change)
        - Data loss segments   (tracker loses the eye → flat line)

    Args:
        signal: Clean 1-D float32 array.
        rng:    NumPy random generator.

    Returns:
        noisy: Corrupted copy of the input signal.
    """
    if rng is None:
        rng = np.random.default_rng()

    noisy = signal.copy()
    n = len(signal)

    # 1. Gaussian noise (always present)
    noise_level = rng.uniform(0.01, 0.04)
    noisy += rng.normal(0, noise_level, n).astype(np.float32)

    # 2. Blink artifacts (0–3 blinks)
    n_blinks = rng.integers(0, 4)
    for _ in range(n_blinks):
        blink_pos    = rng.integers(0, n - 10)
        blink_len    = rng.integers(5, 20)
        blink_end    = min(blink_pos + blink_len, n)
        # Smooth dip using a half-cosine
        dip = np.cos(np.linspace(0, np.pi, blink_end - blink_pos)) * 0.5 + 0.5
        noisy[blink_pos:blink_end] *= (dip * 0.1 + 0.05).astype(np.float32)

    # 3. Spike artifacts (0–5 spikes)
    n_spikes = rng.integers(0, 6)
    spike_idx = rng.integers(0, n, size=n_spikes)
    spike_val = rng.choice([-1, 1], size=n_spikes) * rng.uniform(0.3, 0.8, size=n_spikes)
    noisy[spike_idx] += spike_val.astype(np.float32)

    # 4. Slow baseline drift
    if rng.random() < 0.5:
        drift_amp   = rng.uniform(0.05, 0.15)
        drift_freq  = rng.uniform(0.5, 2.0)
        t           = np.linspace(0, 2 * np.pi * drift_freq, n)
        noisy      += (drift_amp * np.sin(t)).astype(np.float32)

    # 5. Data-loss flat segments
    n_losses = rng.integers(0, 3)
    for _ in range(n_losses):
        loss_pos = rng.integers(0, n - 5)
        loss_len = rng.integers(3, 15)
        loss_end = min(loss_pos + loss_len, n)
        noisy[loss_pos:loss_end] = rng.uniform(0.0, 0.1)

    return noisy


def generate_dataset(
    n_samples: int = 1000,
    signal_length: int = 300,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Generate a paired (noisy, clean) dataset.

    Args:
        n_samples:     Number of signal pairs to generate.
        signal_length: Length of each time series.
        seed:          Random seed for reproducibility.

    Returns:
        noisy_data: float32 array of shape (n_samples, signal_length)
        clean_data: float32 array of shape (n_samples, signal_length)
    """
    rng = np.random.default_rng(seed)
    clean_data = np.stack([
        generate_pupil_signal(signal_length, rng=rng) for _ in range(n_samples)
    ])
    noisy_data = np.stack([
        add_artifacts(clean_data[i], rng=rng) for i in range(n_samples)
    ])
    return noisy_data.astype(np.float32), clean_data.astype(np.float32)


# ─────────────────────────────────────────────
# 2. PYTORCH DATASET
# ─────────────────────────────────────────────

class PupilDataset(Dataset):
    """
    Wraps (noisy, clean) numpy arrays into a PyTorch Dataset.

    Each item is a tuple (noisy_tensor, clean_tensor) of shape (1, L)
    where the channel dimension is required by the CNN.
    """

    def __init__(self, noisy: np.ndarray, clean: np.ndarray):
        assert noisy.shape == clean.shape
        # Add channel dimension: (N, L) → (N, 1, L)
        self.noisy = torch.from_numpy(noisy[:, None, :])
        self.clean = torch.from_numpy(clean[:, None, :])

    def __len__(self) -> int:
        return len(self.noisy)

    def __getitem__(self, idx):
        return self.noisy[idx], self.clean[idx]


# ─────────────────────────────────────────────
# 3. CNN MODEL
# ─────────────────────────────────────────────

class ResidualBlock(nn.Module):
    """1-D residual block with two conv layers and a skip connection."""

    def __init__(self, channels: int, kernel_size: int = 7):
        super().__init__()
        pad = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))


class PupilDenoisingCNN(nn.Module):
    """
    1-D CNN for pupil signal denoising.

    Architecture (encoder–bottleneck–decoder with skip connections):
        Input  (1, L)
        → Encoder: progressively increase channels & receptive field
        → Bottleneck: deep residual processing
        → Decoder: mirror encoder, recover signal length
        → Output (1, L)  — same size as input

    The network learns a residual: output = input + predicted_correction,
    which is easier to learn when the clean signal is close to the noisy one.
    """

    def __init__(self, base_channels: int = 32, n_residual_blocks: int = 4):
        super().__init__()

        c = base_channels

        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv1d(1,     c,    kernel_size=7, padding=3),
            nn.BatchNorm1d(c), nn.ReLU(inplace=True),
        )
        self.enc2 = nn.Sequential(
            nn.Conv1d(c,     c*2,  kernel_size=7, padding=3),
            nn.BatchNorm1d(c*2), nn.ReLU(inplace=True),
        )
        self.enc3 = nn.Sequential(
            nn.Conv1d(c*2,   c*4,  kernel_size=5, padding=2),
            nn.BatchNorm1d(c*4), nn.ReLU(inplace=True),
        )

        # Bottleneck residual blocks
        self.bottleneck = nn.Sequential(
            *[ResidualBlock(c*4, kernel_size=5) for _ in range(n_residual_blocks)]
        )

        # Decoder (mirror encoder)
        self.dec3 = nn.Sequential(
            nn.Conv1d(c*4,   c*2,  kernel_size=5, padding=2),
            nn.BatchNorm1d(c*2), nn.ReLU(inplace=True),
        )
        self.dec2 = nn.Sequential(
            nn.Conv1d(c*2*2, c,    kernel_size=7, padding=3),  # *2 for skip connection
            nn.BatchNorm1d(c), nn.ReLU(inplace=True),
        )
        self.dec1 = nn.Sequential(
            nn.Conv1d(c*2,   c,    kernel_size=7, padding=3),  # *2 for skip connection
            nn.BatchNorm1d(c), nn.ReLU(inplace=True),
        )

        # Final projection → residual correction
        self.head = nn.Conv1d(c, 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder
        e1 = self.enc1(x)    # (B, c,   L)
        e2 = self.enc2(e1)   # (B, c*2, L)
        e3 = self.enc3(e2)   # (B, c*4, L)

        # Bottleneck
        b = self.bottleneck(e3)  # (B, c*4, L)

        # Decoder with skip connections
        d3 = self.dec3(b)                         # (B, c*2, L)
        d2 = self.dec2(torch.cat([d3, e2], dim=1)) # (B, c,   L)
        d1 = self.dec1(torch.cat([d2, e1], dim=1)) # (B, c,   L)

        # Residual output: learn the correction to add to the noisy input
        correction = self.head(d1)
        return x + correction


# ─────────────────────────────────────────────
# 4. TRAINING UTILITIES
# ─────────────────────────────────────────────

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for noisy, clean in loader:
        noisy, clean = noisy.to(device), clean.to(device)
        optimizer.zero_grad()
        pred = model(noisy)
        loss = criterion(pred, clean)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(noisy)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    for noisy, clean in loader:
        noisy, clean = noisy.to(device), clean.to(device)
        pred = model(noisy)
        total_loss += criterion(pred, clean).item() * len(noisy)
    return total_loss / len(loader.dataset)


def train(
    model,
    train_loader,
    val_loader,
    n_epochs: int = 50,
    lr: float = 1e-3,
    device: str = "cpu",
    patience: int = 10,
):
    """
    Train the model with early stopping.

    Returns:
        history: dict with 'train_loss' and 'val_loss' lists.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5, verbose=True
    )
    criterion = nn.MSELoss()

    history = {"train_loss": [], "val_loss": []}
    best_val, no_improve = float("inf"), 0

    for epoch in range(1, n_epochs + 1):
        tr_loss  = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{n_epochs} | train {tr_loss:.5f} | val {val_loss:.5f}")

        # Early stopping
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            no_improve = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(torch.load("best_model.pt"))
    return history


# ─────────────────────────────────────────────
# 5. REAL-WORLD DATA STRUCTURE
# ─────────────────────────────────────────────

class RealWorldPupilDataset(Dataset):
    """
    Dataset for real-world pupil recordings (no clean ground truth).

    Pass your real measurements as a 2-D numpy array: (n_trials, signal_length).
    Use train_test_split() below to get train/test subsets.

    Example usage:
        raw = np.load("my_pupil_data.npy")          # shape (N, 300)
        train_ds, test_ds = split_real_data(raw)
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    """

    def __init__(self, signals: np.ndarray):
        # signals: (N, L) float32
        self.signals = torch.from_numpy(signals[:, None, :].astype(np.float32))

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        # Returns only the noisy signal (no ground truth for real data)
        return self.signals[idx]


def split_real_data(
    signals: np.ndarray,
    test_size: float = 0.2,
    seed: int = 42,
) -> tuple[RealWorldPupilDataset, RealWorldPupilDataset]:
    """
    Split real-world signals into train / test datasets.

    Args:
        signals:   2-D array of shape (n_trials, signal_length).
        test_size: Fraction of data for the test set (default 20 %).
        seed:      Random seed.

    Returns:
        train_dataset, test_dataset
    """
    train_signals, test_signals = train_test_split(
        signals, test_size=test_size, random_state=seed
    )
    return RealWorldPupilDataset(train_signals), RealWorldPupilDataset(test_signals)


# ─────────────────────────────────────────────
# 6. VISUALISATION HELPER
# ─────────────────────────────────────────────

def plot_predictions(model, noisy_np, clean_np, n_examples=3, device="cpu"):
    """
    Plot noisy input, model prediction, and ground truth side-by-side.

    Args:
        noisy_np: (N, L) numpy array of noisy signals.
        clean_np: (N, L) numpy array of clean signals.
    """
    model.eval()
    indices = np.random.choice(len(noisy_np), n_examples, replace=False)

    fig, axes = plt.subplots(n_examples, 1, figsize=(12, 3 * n_examples))
    if n_examples == 1:
        axes = [axes]

    with torch.no_grad():
        for ax, idx in zip(axes, indices):
            x = torch.from_numpy(noisy_np[idx][None, None, :]).to(device)
            pred = model(x).cpu().numpy()[0, 0]
            ax.plot(noisy_np[idx], alpha=0.5, color="steelblue", label="Noisy input")
            ax.plot(clean_np[idx], color="green",  lw=2, label="Ground truth")
            ax.plot(pred,          color="tomato",  lw=2, linestyle="--", label="Model output")
            ax.set_ylabel("Pupil size")
            ax.legend(loc="upper right", fontsize=8)
            ax.set_title(f"Sample {idx}")

    axes[-1].set_xlabel("Time (samples)")
    plt.suptitle("Pupil Signal Denoising", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("predictions.png", dpi=150)
    plt.show()
    print("Saved predictions.png")

# ─────────────────────────────────────────────
# 7. MAIN — end-to-end demo
# ─────────────────────────────────────────────

if __name__ == "__main__":
    # ── Config ────────────────────────────────
    N_SAMPLES     = 2000    # synthetic training pairs
    SIGNAL_LENGTH = 300     # samples per trial
    BATCH_SIZE    = 64
    N_EPOCHS      = 60
    LR            = 1e-3
    DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {DEVICE}")

    # ── Step 1: Generate synthetic pre-training data ──
    print("\n[1/4] Generating synthetic data...")
    noisy_all, clean_all = generate_dataset(
        n_samples=N_SAMPLES,
        signal_length=SIGNAL_LENGTH,
        seed=42,
    )

    # 80/20 train/val split for pre-training
    split = int(0.8 * N_SAMPLES)
    train_ds  = PupilDataset(noisy_all[:split],  clean_all[:split])
    val_ds    = PupilDataset(noisy_all[split:],  clean_all[split:])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"  Train: {len(train_ds)} samples | Val: {len(val_ds)} samples")

    # ── Step 2: Build & train the CNN ────────
    print("\n[2/4] Building model...")
    model = PupilDenoisingCNN(base_channels=32, n_residual_blocks=4)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,}")

    print("\n[3/4] Pre-training on synthetic data...")
    history = train(model, train_loader, val_loader,
                    n_epochs=N_EPOCHS, lr=LR, device=DEVICE, patience=10)

    # ── Step 3: Real-world data structure ────
    print("\n[4/4] Demonstrating real-world data split...")
    # Replace this with: real_data = np.load("your_data.npy")
    real_data = np.random.randn(200, SIGNAL_LENGTH).astype(np.float32) * 0.1 + 0.8
    train_real_ds, test_real_ds = split_real_data(real_data, test_size=0.2)
    print(f"  Real-world → Train: {len(train_real_ds)}, Test: {len(test_real_ds)}")

    # Fine-tune on real labelled data if you have (noisy, clean) pairs:
    # real_noisy = ...   shape (N, SIGNAL_LENGTH)
    # real_clean = ...   shape (N, SIGNAL_LENGTH)
    # real_train_ds = PupilDataset(real_noisy_train, real_clean_train)
    # history_ft = train(model, real_train_loader, real_val_loader, ...)

    # ── Visualise ────────────────────────────
    plot_predictions(model, noisy_all[split:split+10],
                     clean_all[split:split+10], n_examples=3, device=DEVICE)

    # Plot training curves
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_loss"], label="Train loss")
    plt.plot(history["val_loss"],   label="Val loss")
    plt.xlabel("Epoch"); plt.ylabel("MSE loss")
    plt.title("Training history"); plt.legend()
    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150)
    plt.show()
    print("Saved training_history.png")
